# 🚀 MyBabyAI (CodeMind) - Turbo Cloud Training

Bu notebook, CodeMind modellerini **Google Colab** veya **Kaggle** üzerinde **maksimum hızda** eğitmek için tasarlanmıştır.

Orijinal `cloud_train.ipynb` ile birebir aynıdır, ancak ek olarak:
- 🎛️ **GPU Profil Seçici** — 16GB / 24GB / 40GB / 80GB için otomatik ayar
- ⚡ **Flash Attention 2** — Ampere+ GPU'larda otomatik aktif
- 🔥 **Torch Compile** — ~%15-20 ekstra hız
- 🎯 **BF16 + TF32** — Daha stabil ve hızlı eğitim
- 📏 **Uzun Sekans Desteği** — 4096'ya kadar (Flash Attention gerektirir)

In [ ]:
# @title 1. 🛠️ SİSTEM VE BAĞIMLILIKLARI KUR

# ─── ⚡ PYTORCH OOM ÖNLEYİCİ (torch import'tan ÖNCE set edilmeli) ───
import os, sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('✅ PYTORCH_CUDA_ALLOC_CONF set edildi.')

# Ortam Algılama
ENV = "colab" if "google.colab" in sys.modules else "kaggle" if os.path.exists("/kaggle") else "local"
print(f"Detected Environment: {ENV.upper()}")

# Paket Kurulumu — yerel ortamda zaten kurulu, sadece cloud'da çalıştır
if ENV != "local":
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q",
        "transformers", "peft", "bitsandbytes>=0.41.0", "sentencepiece", "huggingface_hub"])
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q",
        "pyyaml", "python-dotenv", "rich", "tqdm", "psutil", "requests",
        "datasets", "chromadb", "sentence-transformers"])
    print("✅ Paket kurulumu tamamlandı.")
else:
    print("ℹ️ Yerel ortam — pip kurulumu atlandı.")

# HuggingFace Token — Python API ile login (shell değişken sorunu olmaz)
HF_TOKEN = None
if ENV == "colab":
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
elif ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
else:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ HuggingFace Girişi Yapıldı")
else:
    print("ℹ️ HF_TOKEN bulunamadı, giriş atlandı.")

In [ ]:
# @title 2. 📂 REPO VE DİZİN AYARLARI
REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

import os, sys

if ENV == "local":
    print("🏠 Yerel ortam algılandı, git işlemleri ve dizin değişimi atlanıyor.")
else:
    if "google.colab" in sys.modules:
        os.chdir("/content")
    elif os.path.exists("/kaggle/working"):
        os.chdir("/kaggle/working")
    
    if not os.path.exists("mybabyai"):
        !git clone -b {BRANCH} {REPO_URL}
    %cd mybabyai
    !git fetch --all
    !git reset --hard origin/{BRANCH}
    !git pull

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.makedirs("models/fine_tuned", exist_ok=True)
print("✅ Çalışma dizini hazır:", os.getcwd())


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# @title 2.5. 💾 CHECKPOINT YÜKLE (G-Drive'dan Aktarma)
import os

GDRIVE_FILE_ID = "" # @param {type:"string"}

if GDRIVE_FILE_ID:
    print("⏳ Checkpoint indiriliyor...")
    %pip install -q gdown
    !gdown --id {GDRIVE_FILE_ID} -O checkpoint.zip
    !unzip -q checkpoint.zip -d checkpoints/
    !rm checkpoint.zip
    print("✅ Checkpoint çalışma dizinine aktarıldı.")
else:
    print("ℹ️ G-Drive ID girilmedi, eğer checkpoint'iniz varsa kendiniz yükleyin.")

In [ ]:
# @title 3. 🎛️ GPU PROFİL SEÇİCİ — Turbo Hızlandırma
import torch

# ─── GPU Profilinizi Seçin ───────────────────────────────────────────────────
# "AUTO"   → GPU'yu otomatik algılar, en iyi profili atar
# "16GB"   → T4 / RTX3080 / P100 (Colab Ücretsiz)
# "24GB"   → RTX3090 / RTX4090 / A5000
# "40GB"   → A100-SXM4-40GB (Colab Pro+, Kaggle)
# "80GB"   → A100-SXM4-80GB / H100 (Bulut Sunucu)
GPU_PROFILE = "AUTO" # @param ["AUTO", "16GB", "24GB", "40GB", "80GB"]

# ─── Turbo Özellikleri ────────────────────────────────────────────────────────
ENABLE_TORCH_COMPILE = True  # @param {type:"boolean"} ~%15-20 hız kazancı (ilk epoch yavaş olabilir)
ENABLE_BF16         = True  # @param {type:"boolean"} Daha stabil, FP16'dan daha iyi (Ampere+)

# ─── Profil Tablosu ───────────────────────────────────────────────────────────
PROFILES = {
    "16GB":  {"batch": 1,  "grad_acc": 16, "seq": 512,  "optimizer": "adamw_torch",       "label": "T4 / P100"},
    "24GB":  {"batch": 4,  "grad_acc": 8,  "seq": 1024, "optimizer": "adamw_torch_fused", "label": "RTX3090 / RTX4090"},
    "40GB":  {"batch": 8,  "grad_acc": 4,  "seq": 2048, "optimizer": "adamw_torch_fused", "label": "A100-40GB"},
    "80GB":  {"batch": 16, "grad_acc": 4,  "seq": 4096, "optimizer": "adamw_torch_fused", "label": "A100-80GB / H100"},
}

# ─── AUTO Algılama ────────────────────────────────────────────────────────────
if GPU_PROFILE == "AUTO":
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb >= 70:
            GPU_PROFILE = "80GB"
        elif vram_gb >= 35:
            GPU_PROFILE = "40GB"
        elif vram_gb >= 20:
            GPU_PROFILE = "24GB"
        else:
            GPU_PROFILE = "16GB"
        print(f"🔍 AUTO: {vram_gb:.1f}GB VRAM tespit edildi → Profil: {GPU_PROFILE}")
    else:
        GPU_PROFILE = "16GB"
        print("⚠️ CUDA bulunamadı, varsayılan 16GB profili kullanılıyor.")

profile = PROFILES[GPU_PROFILE]

# Flash Attention: Ampere+ GPU'larda (SM>=80) otomatik aktif
USE_FLASH_ATTN = "eager"
if torch.cuda.is_available():
    sm_major = torch.cuda.get_device_capability()[0]
    sm_minor = torch.cuda.get_device_capability()[1]
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9

    if sm_major >= 8:  # Ampere+ (A100, RTX3090, RTX4090, H100)
        USE_FLASH_ATTN = "flash_attention_2"
        print(f"✅ Ampere+ GPU: Flash Attention 2 aktif")
        # TF32
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("⚡ TF32 Enabled")
    elif sm_major >= 7:  # Volta/Turing (T4, V100)
        USE_FLASH_ATTN = "sdpa"
        print(f"✅ Turing GPU: SDPA aktif")
    else:  # Pascal ve altı (P100)
        USE_FLASH_ATTN = "sdpa"
        ENABLE_BF16 = False  # Pascal BF16 desteklemez
        print(f"⚠️ Pascal GPU (P100): SDPA aktif, BF16 devre dışı")

    print(f"\n{'─'*55}")
    print(f"🎛️  GPU PROFİL: {GPU_PROFILE} ({profile['label']})")
    print(f"   GPU        : {gpu_name} — {total_vram:.1f} GB VRAM")
    print(f"   Batch Size : {profile['batch']} | Grad Acc: {profile['grad_acc']} | Eff. Batch: {profile['batch']*profile['grad_acc']}")
    print(f"   Seq Length : {profile['seq']}")
    print(f"   Optimizer  : {profile['optimizer']}")
    print(f"   Attention  : {USE_FLASH_ATTN}")
    print(f"   Torch Comp.: {'✅' if ENABLE_TORCH_COMPILE else '❌'}")
    print(f"   BF16       : {'✅' if ENABLE_BF16 else '❌'}")
    print(f"{'─'*55}")

In [ ]:
# @title 4. ⚙️ EĞİTİM KONFİGÜRASYONU

model_size        = "650M"  # @param ["10M", "20M", "125M", "350M", "400M", "350M-MoE", "650M"]
training_mode     = "full"  # @param ["lora", "full"]
load_from_checkpoint  = False  # @param {type:"boolean"}
resume_from_checkpoint = False  # @param {type:"boolean"}
pretrained_tokenizer = "ytu-ce-cosmos/Turkish-GPT2-large" # @param {type:"string"}

# ─── Profil değerlerini ata (değiştirmek istersen buradan override edebilirsin) ───
override_profile      = False # @param {type:"boolean"}
batch_size           = 16    # @param {type:"integer"}
gradient_accumulation = 1 # @param {type:"integer"}
max_seq_length       = 2048    # @param {type:"integer"}

if not override_profile:
    batch_size = profile["batch"]
    gradient_accumulation = profile["grad_acc"]
    max_seq_length = profile["seq"]

learning_rate     = 1e-4   # @param {type:"number"}
lr_scheduler_type = "cosine" # @param ["cosine", "linear", "constant"]
warmup_steps      = 500    # @param {type:"integer"}
epochs            = 3      # @param {type:"integer"}
logging_steps     = 1      # @param {type:"integer"}
max_steps         = -1     # @param {type:"integer"} (-1 = epoch bazlı)
save_steps        = 500    # @param {type:"integer"}
hf_repo           = ""     # @param {type:"string"}

if ENV == "colab":
    import glob
    if glob.glob('./checkpoint-*') or glob.glob('/content/checkpoint-*'):
        output_dir = '.' if glob.glob('./checkpoint-*') else '/content'
        print(f"⚠️ Manuel checkpoint tespiti. output_dir = {output_dir}")
    else:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            try:
                drive.mount("/content/drive")
            except Exception:
                print("⚠️ Google Drive mount desteklenmiyor.")
        output_dir = "/content/drive/MyDrive/mybabyai_checkpoints" if os.path.exists("/content/drive") else "checkpoints"
else:
    output_dir = "checkpoints"

os.makedirs(output_dir, exist_ok=True)

print(f"--- {model_size} ({training_mode}) Yapılandırması Tamam ---")
print(f"    Batch: {batch_size} | GradAcc: {gradient_accumulation} | EffBatch: {batch_size*gradient_accumulation}")
print(f"    MaxSeqLen: {max_seq_length} | LR: {learning_rate} | Epochs: {epochs}")
print(f"    OutputDir: {output_dir}")

In [ ]:
# # @title 4.5. ⚡ FLASH ATTENTION 2 KURULUMU (Ampere+ GPU'lar için)
# # Bu hücre sadece Flash Attention 2 kullanılacaksa çalıştırılmalıdır.
# # A100, RTX3090, RTX4090, H100 gibi GPU'larda ~%30-40 hız kazancı sağlar.
# # Kurulum 10-25 dakika sürebilir. T4/P100'de SDPA kullanıldığı için bu adım gerekmez.

# if USE_FLASH_ATTN == "flash_attention_2":
#     print("⚡ Flash Attention 2 kuruluyor (bu işlem 10-25 dakika sürebilir)...")
#     !pip install -q flash-attn --no-build-isolation --verbose
#     print("✅ Flash Attention 2 kurulumu tamamlandı.")
# else:
#     print(f"ℹ️ GPU profiliniz SDPA kullanıyor ({USE_FLASH_ATTN}). Flash-attn kurulumu atlandı.")

In [ ]:
# @title 5. 🤖 MODEL & TRAINER BAŞLATMA
import gc
import torch

# ── VRAM temizliği ──
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | {total_gb:.1f} GB toplam | {free_gb:.1f} GB boş")

from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
config.set("training.output_dir", output_dir)
config.set("training.per_device_train_batch_size", batch_size)
config.set("training.gradient_accumulation_steps", gradient_accumulation)
config.set("training.learning_rate", learning_rate)
config.set("training.lr_scheduler_type", lr_scheduler_type)
config.set("training.warmup_steps", warmup_steps)
config.set("training.num_train_epochs", epochs)
config.set("training.save_steps", save_steps)
config.set("training.resume_from_checkpoint", resume_from_checkpoint)
config.set("training.max_length", max_seq_length)
config.set("training.gradient_checkpointing", True)
config.set("model.attn_implementation", USE_FLASH_ATTN)

# ─── Turbo Optimizasyonları ───────────────────────────────────────────────────
config.set("training.optim", profile["optimizer"])
config.set("training.bf16", ENABLE_BF16)
config.set("training.tf32", True)
config.set("training.torch_compile", ENABLE_TORCH_COMPILE)
config.set("training.dataloader_num_workers", 4)
config.set("training.dataloader_pin_memory", True)
# ─────────────────────────────────────────────────────────────────────────────

if hf_repo:
    config.set("training.push_to_hub", True)
    config.set("training.hub_model_id", hf_repo)
    config.set("training.hub_strategy", "every_save")

if pretrained_tokenizer:
    config.set("model.pretrained_tokenizer", pretrained_tokenizer)

model_manager = ModelManager(config)

if resume_from_checkpoint:
    print(f"\n🔄 Eğitim {output_dir} dizinindeki checkpoint'ten devam edecek.")
    model_manager.load_fresh_model(size=model_size)
elif load_from_checkpoint:
    print("\n📦 Mevcut base-model checkpoint yükleniyor...")
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    print(f"\n🌱 Sıfırdan CodeMind-{model_size} oluşturuluyor...")
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)
print(f"✅ {model_manager.model_name} Eğitime Hazır!")

In [ ]:
# @title 6. 🎯 DATASET HAVUZU (TÜRKÇE ODAKLI)

# max_samples GPU profiline göre otomatik ölçeklenir
_samples_scale = {"16GB": 1000, "24GB": 3000, "40GB": 5000, "80GB": 10000}
_base = _samples_scale.get(GPU_PROFILE, 1000)

dataset_pool = [
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇹🇷 GPT-4 Alpaca TR",
        "type": "huggingface",
        "dataset_key": "alpaca_gpt4_tr",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇹🇷 OpenOrca TR",
        "type": "huggingface",
        "dataset_key": "open_orca_tr",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇬🇧 Slim Orca (EN)",
        "type": "huggingface",
        "dataset_key": "slim_orca",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇬🇧 WizardLM (EN, Logic/Code)",
        "type": "huggingface",
        "dataset_key": "wizardlm_70k",
        "streaming": True,
        # "max_samples": _base
    },
    {
        "name": "🇹🇷 BellaTurca (Large Scale)",
        "type": "huggingface",
        "dataset_key": "bellaturca",
        "streaming": True,
        "lazy_load": True
    },
]

print(f"📊 Veri havuzu hazır: {len(dataset_pool)} kaynak.")
print(f"   Not: Tüm veriler (full dataset) havuzdan çekiliyor. Sınır kaldırıldı.")


In [ ]:
# @title 7. 🚀 EĞİTİMİ BAŞLAT
import gc, torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f"🔋 Eğitim öncesi boş VRAM: {free_gb:.1f} GB")

try:
    trainer.train_from_pool(
        dataset_pool,
        training_type=training_mode,
        max_length=max_seq_length,
        max_steps=max_steps,
        use_notebook_callback=True,
        resume_from_checkpoint=resume_from_checkpoint,
    )
except KeyboardInterrupt:
    print("\n🛑 Eğitim kullanıcı tarafından durduruldu.")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("\n❌ CUDA OOM! Öneri:")
        print("   1. Hücre 3'te daha düşük bir GPU profili seç (örn: 16GB yerine 24GB varsa)")
        print("   2. max_seq_length'i yarıya indir ve yeniden başlat")
        print("   3. training_mode='lora' yap")
        print("   4. Kernel'i Restart edip Hücre 1'den başa al")
        torch.cuda.empty_cache()
        gc.collect()
    raise

## 📖 GPU Profil Rehberi

| Profil | GPU | Batch | Seq Len | Optimizer | Flash Attn |
|--------|-----|-------|---------|-----------|------------|
| **16GB** | T4 / P100 | 1 | 512 | adamw_torch | SDPA |
| **24GB** | RTX3090 / RTX4090 | 4 | 1024 | adamw_torch_fused | Flash Attn 2 |
| **40GB** | A100-40GB | 8 | 2048 | adamw_torch_fused | Flash Attn 2 |
| **80GB** | A100-80GB / H100 | 16 | 4096 | adamw_torch_fused | Flash Attn 2 |

### OOM Hatası Alırsanız:
| Adım | Aksiyon |
|------|----------|
| 1 | Hücre 3'te bir alt profil seç |
| 2 | `max_seq_length`'i **128** yap (en etkili) |
| 3 | `gradient_accumulation`'ı **32**'ye yükselt |
| 4 | `training_mode='lora'` yap |
| 5 | Kernel'i **Restart** edip Hücre 1'den başa al |

> **⚠️ Not:** P100 (Pascal, SM60) üzerinde `ENABLE_BF16` otomatik kapatılır ve `paged_adamw_8bit` çalışmaz.
> P100 için `training_mode='lora'` ve `max_seq_length=128` önerilir.